In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('appartments_cleaned.csv').drop(22)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 245 entries, 0 to 245
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   PropertyName        245 non-null    object
 1   PropertySubName     245 non-null    object
 2   NearbyLocations     245 non-null    object
 3   LocationAdvantages  245 non-null    object
 4   Link                245 non-null    object
 5   PriceDetails        245 non-null    object
 6   TopFacilities       245 non-null    object
 7   FacilitiesStr       245 non-null    object
dtypes: object(8)
memory usage: 17.2+ KB


In [ ]:
import ast

all_locations = []

for item in df['LocationAdvantages'].dropna():
    d = ast.literal_eval(item)
    all_locations.extend(d.keys())

print(len(all_locations))

2546


In [ ]:
import re

def clean_location(x):
    x = x.lower().strip()

    x = x.replace('.', '')
    x = x.replace(',', '')
    x = x.replace('-', ' ')
    x = x.replace('_', ' ')

    x = re.sub(r'\s+', ' ', x)

    return x

clean_locations = [clean_location(i) for i in all_locations]

In [ ]:
from collections import Counter

counter = Counter(clean_locations)

counter.most_common(50)

[('indira gandhi international airport', 87),
 ('indira gandhi intl airport', 70),
 ('dwarka expressway', 54),
 ('park hospital', 32),
 ('gurgaon railway station', 32),
 ('garhi harsaru junction', 32),
 ('skyjumper trampoline park', 32),
 ('gurugram university', 29),
 ('sushant university', 28),
 ('igi airport', 26),
 ('the northcap university', 25),
 ('sector 55 56 metro station', 24),
 ('imt manesar', 21),
 ('dlf golf and country club', 20),
 ('dwarka expy', 19),
 ('sapphire 83 mall', 19),
 ('dpg institute of technology', 19),
 ('airia mall', 19),
 ('tau devilal sports complex', 18),
 ('de adventure park', 17),
 ('sohna road', 17),
 ('badshahpur sohna rd hwy', 17),
 ('holiday inn gurugram sector 90', 16),
 ('huda metro station', 14),
 ('sgt university', 14),
 ('nh 48', 13),
 ('euro international school', 13),
 ('hyatt regency gurgaon', 13),
 ('pataudi road', 12),
 ('pvr drive in theatre', 12),
 ('golf course extension road', 11),
 ('nakhrola stadium', 11),
 ("oyster's water park", 11

In [ ]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.5 MB/s eta 0:00:00


In [ ]:
from rapidfuzz import process, fuzz

unique_locations = list(set(clean_locations))

mapping = {}

threshold = 90

for loc in unique_locations:

    if loc in mapping:
        continue

    matches = process.extract(
        loc,
        unique_locations,
        scorer=fuzz.token_sort_ratio,
        limit=None
    )

    for match, score, _ in matches:

        if score >= threshold:
            mapping[match] = loc

In [ ]:
pd.DataFrame(mapping.items(), columns=["Original", "Mapped"])

In [ ]:
clean_locations = [
    mapping.get(i, i)
    for i in clean_locations
]

In [ ]:
len(set(clean_locations))

938

In [ ]:
sorted(set(clean_locations))

In [ ]:
keywords = [
    "airport",
    "hospital",
    "metro",
    "mall",
    "school",
    "road",
    "sector",
    "park",
    "club",
    "college",
    "university"
]

for word in keywords:
    print("\n", "="*40)
    print(word.upper())
    print("="*40)

    for loc in sorted(set(clean_locations)):
        if word in loc:
            print(loc)

In [ ]:
manual_map = {

    # ---------------- AIRPORT ----------------
    "ig international airport": "indira gandhi international airport",
    "igia airport": "indira gandhi international airport",
    "indira gandhi int airport": "indira gandhi international airport",
    "airport": "indira gandhi international airport",

    # ---------------- HOSPITAL ----------------
    "aartemis hospital gurgaon": "artemis hospital",

    "ck birla hospital gurgaon": "ck birla hospital",

    "paras hospitals": "paras hospital",
    "paras hospitals gurgaon": "paras hospital",

    "manipal hospital gurugram": "manipal hospital",
    "manipal hospital palam vihar": "manipal hospital",

    "miracles apollo cradle hospital": "miracles apollo cradle /spectra hospital",
    "miracles apollo cradle spectra hospital": "miracles apollo cradle /spectra hospital",
    "miracles apollo cradle/spectra hospital": "miracles apollo cradle /spectra hospital",

    "park hospital palam vihar": "park hospital",

    "sanjeevani hospital child specialist": "sanjeevani hospital",

    "signature advanced hospital": "signature hospital",
    "the signature super speciality hospital": "signature hospital",

    "silver streak multi speciality hospital": "silver streak hospital",

    "vardaan hospital and trauma centre": "vardaan hospital",

    # ---------------- METRO ----------------
    "huda metro station (gurugram)": "huda metro station",

    "sector 42 43 metro station": "sector 42 43 rapid metro station",
    "sector 42 43 rapid metro": "sector 42 43 rapid metro station",

    "sector 55 56 metro": "sector 55 56 rapid metro station",
    "sector 55 56 rapid metro": "sector 55 56 rapid metro station",
    "sector 55/56 metro station": "sector 55 56 rapid metro station",

    "sector 54 chowk": "sector 54 chowk metro station",

    # ---------------- MALL ----------------
    "airia mall sector 68": "airia mall",

    "ambience mall new": "ambience mall",

    "global foyer mall palam vihar": "global foyer mall",

    "paras trinity shopping mall": "paras trinity mall",

    "sapphire 83 mall sector 83": "sapphire 83 mall",

    "the hive shopping mall": "the hive mall",

    # ---------------- SCHOOL ----------------
    "alpine convent school": "alpine school",

    "delhi public school gurugram sector 67a": "delhi public school",
    "delhi public school sector 103": "delhi public school",

    "euro int school": "euro international school",
    "euro intl school sec 51": "euro international school",
    "euro intl school sector 109": "euro international school",
    "euro intl school sector 37d gurugram": "euro international school",

    "gd goenka high school": "gd goenka school",
    "gd goenka public school": "gd goenka school",
    "gd goenka signature school": "gd goenka school",

    "heritage intl experiential school": "heritage xperiential learning school",

    "lotus valley intl school gurgaon": "lotus valley international school",

    "narayana e techno school manesar": "narayana e techno school",

    "pranavananda int school": "pranavananda international school",

    "prime scholars int school": "prime scholars international school",

    "suncity school gurgaon": "suncity school",
    "suncity school sector 37d": "suncity school",

    "st pauls school": "saint paul's school",

    "vega schools nh 8": "vega school",
    "vega schools sector 48": "vega school",

    # ---------------- ROAD ----------------
    "gurgaon road": "gurugram road",

    "mg road metro station": "mg road",

    "radisson hotel gurugram sohna road": "radisson hotel sohna road",

    # ---------------- BUSINESS PARK ----------------
    "vatika business park sector 49": "vatika business park",
    "vatika business park sohna rd": "vatika business park",

    # ---------------- CLUB ----------------
    "aipl business club sector 62": "aipl business club",

    # ---------------- COLLEGE ----------------
    "dpgitm engineering college sector 34": "dpgitm engineering college",

    "kiit college of engineering": "kiit college",

    "suraj pg degree college sec 75": "suraj pg degree college",

    # ---------------- UNIVERSITY ----------------
    "amity university gurugram": "amity university",

    "gd goenka university gurugram": "gd goenka university",

    "gurugram university kankrola": "gurugram university",
    "gurugram university sector 87": "gurugram university",

    "iilm university gurugram": "iilm university",

    "kr mangalam university sohna": "kr mangalam university",

    "genesis hospital sector 84": "genesis hospital",
    "park inn gurgaon": "park inn",
    "holiday inn gurugram sector 90": "holiday inn sector 90",
    "aarvy healthcare" : 'aarvy healthcare super speciality hospital',
    'aarvy healthcare hospital' : 'aarvy healthcare super speciality hospital',
    'aarvy healthcare super speciality' : 'aarvy healthcare super speciality hospital',
    'adarsh public schoolgarhi harsaru' : 'adarsh public school garhi harsaru',
    'amity' : 'amity university',
    'appu ghar': 'appu ghar water park',
    'aravali adventure hill' : 'aravalli hill view point',
    'aravalli hills' : 'aravalli hill view point',
    'arc hospital' : 'arc multi speciality hospital',
    'arc multi speciality' : 'arc multi speciality hospital',
    'artemis hospital gurgaon': 'artimis hospital',
    'ascendas onehub gurgaon' : 'ascendas onehub business park',
    'ascendas onehub gurgaon business park' : 'ascendas onehub business park',
    'bm college of technology' : 'bm college of technology & mgmt',
    'country inn' : 'country inn & suites by radisson',
    'double tree by hilton' : 'doubletree by hilton hotel',
    'doubletree by hilton hotel gurgaon' : 'doubletree by hilton hotel',
    'dpg institute of technology' : 'dpgitm engineering college',
    'dpsg palam vihar gurugram' : 'dpsg palam vihar',
    'ektaa hospitals main sohna rd' : 'ektaa hospitals',
    'emerald plaza shopping mall' : 'emerald plaza',
    'fun n food village' : 'fun n food waterpark',
    'fun n food water park' : 'fun n food waterpark',
    'golden greens golf & resorts' : 'golden greens golf & resorts limited',
    'golf corse ext rd' : 'golf course extension rd',
    'golf course extn road' : 'golf course extension rd',
    'green field school' : 'green field public school',
    'hyatt regency gurgaon' : 'hyatt regency',
    'hyatt regency gurugram' : 'hyatt regency',
    'hyatt regency hotel' : 'hyatt regency',
    'imt manesar gurugram' : 'imt manesar',
     'ion digital zone (gurugram)' : 'ion digital zone gurugram',
    'ion digital zone gurgaon' : 'ion digital zone gurugram',
    'iris broadway gurugram' : 'iris broadway mall',
    'jms marine square' :  'jms marine square mall',
    'kunskapsskolan international': 'kunskapsskolan international school',
    'kunskapsskolan school' : 'kunskapsskolan international school',
    'lotus valley school' : 'lotus valley international school',
    'm3m cosmopolitan' : 'm3m cosmopolitan mall',
    'omaxe city centre' : 'omaxe city centre mall',
    'omaxe gurgaon mall' : 'omaxe city centre mall',
    'paras trinity' : 'paras trinity mall',
    'paras trinity mall sector 63' : 'paras trinity mall',
    'royal institute of science' : 'royal institute of science & management',
    'royal institute of science and mgt' : 'royal institute of science & management',
    'savoy suites manesar' : 'savoy suites',
    'silver streak hospital': 'silver streak multi speciality hospital',
    'skyjumper trampoline park gurgaon' : 'skyjumper trampoline park',
    'sohna rd' : 'sohna road',
    'southern periphery rd' : 'southern periphery road',
    'ss omnia sector 86' : 'ss omnia mall',
    'taj city centre gurugram' : 'taj city centre',
    'taj city centre hotel' : 'taj city centre',
    'the oberoi gurgaon' : 'the oberoi',
    'vipul trade business centre' :  'vipul trade centre',
    'western peripheral expy gurugram' : 'western peripheral expressway',
    'worldmark gurgaon' : 'worldmark',
    'worldmark gurgaon maidawas rd' : 'worldmark'
}


In [ ]:
clean_locations = [
    manual_map.get(i, i)
    for i in clean_locations
]

In [ ]:
len(set(clean_locations))



827

In [ ]:
sorted(set(clean_locations))

In [ ]:
def update_dict(loc_dict):

    new_dict = {}

    for key, value in loc_dict.items():

        key = clean_location(key)

        key = mapping.get(key, key)

        key = manual_map.get(key, key)

        new_dict[key] = value

    return new_dict

In [ ]:
df['LocationAdvantages'] = (
    df['LocationAdvantages']
    .apply(ast.literal_eval)
    .apply(update_dict)
)

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def extract_list(s):
    return re.findall(r"'(.*?)'", s)

df['TopFacilities'] = df['TopFacilities'].apply(extract_list)

In [ ]:
df['FacilitiesStr'] = df['TopFacilities'].apply(' '.join)

In [ ]:
df['FacilitiesStr'][0]

'Swimming Pool Salon Restaurant Spa Cafeteria Sun Deck 24x7 Security Club House Gated Community'

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [ ]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df['FacilitiesStr'])

In [ ]:
tfidf_matrix.toarray()[0]

array([0.        , 0.        , 0.        , 0.1880099 , 0.1880099 ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [ ]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
cosine_sim1.shape

(245, 245)

In [ ]:
df[['PropertyName','PriceDetails']]['PriceDetails'][1]

"{'3 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '1,605 - 2,170 sq.ft.', 'price-range': '₹ 2.2 - 3.03 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,248 - 2,670 sq.ft.', 'price-range': '₹ 3.08 - 3.73 Cr'}}"

In [ ]:
def recommend_properties(property_name, cosine_sim=cosine_sim1):
    # Get the index of the property that matches the name
    idx = df.index[df['PropertyName'] == property_name].tolist()[0]

    # Get the pairwise similarity scores with that property
    sim_scores = list(enumerate(cosine_sim1[idx]))

    # Sort the properties based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar properties
    sim_scores = sim_scores[1:6]

    # Get the property indices
    property_indices = [i[0] for i in sim_scores]

    recommendations_df = pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[property_indices],
        'SimilarityScore': sim_scores
    })

    # Return the top 10 most similar properties
    return recommendations_df

In [ ]:
recommend_properties("DLF The Arbour")

,PropertyName,SimilarityScore
63,Ace Palm Floors,"(62, 0.452815794404316)"
216,Yashika 104,"(215, 0.4198479006103044)"
92,JMS The Nation,"(91, 0.41649165752204825)"
153,India Rashtra,"(152, 0.3988357878631656)"
0,Smartworld One DXP,"(0, 0.38867472309335815)"


In [ ]:

import pandas as pd
import json

# Load the dataset
df_appartments = pd.read_csv('appartments_cleaned.csv').drop(22)

# Function to parse and extract the required features from the PriceDetails column
def refined_parse_modified_v2(detail_str):
    try:
        details = json.loads(detail_str.replace("'", "\""))
    except:
        return {}

    extracted = {}
    for bhk, detail in details.items():
        # Extract building type
        extracted[f'building type_{bhk}'] = detail.get('building_type')

        # Parsing area details
        area = detail.get('area', '')
        area_parts = area.split('-')
        if len(area_parts) == 1:
            try:
                value = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area low {bhk}'] = value
                extracted[f'area high {bhk}'] = value
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None
        elif len(area_parts) == 2:
            try:
                extracted[f'area low {bhk}'] = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area high {bhk}'] = float(area_parts[1].replace(',', '').replace(' sq.ft.', '').strip())
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None

        # Parsing price details
        price_range = detail.get('price-range', '')
        price_parts = price_range.split('-')
        if len(price_parts) == 2:
            try:
                extracted[f'price low {bhk}'] = float(price_parts[0].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                extracted[f'price high {bhk}'] = float(price_parts[1].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                if 'L' in price_parts[0]:
                    extracted[f'price low {bhk}'] /= 100
                if 'L' in price_parts[1]:
                    extracted[f'price high {bhk}'] /= 100
            except:
                extracted[f'price low {bhk}'] = None
                extracted[f'price high {bhk}'] = None

    return extracted
# Apply the refined parsing and generate the new DataFrame structure
data_refined = []

for _, row in df_appartments.iterrows():
    features = refined_parse_modified_v2(row['PriceDetails'])

    # Construct a new row for the transformed dataframe
    new_row = {'PropertyName': row['PropertyName']}

    # Populate the new row with extracted features
    for config in ['1 BHK', '2 BHK', '3 BHK', '4 BHK', '5 BHK', '6 BHK', '1 RK', 'Land']:
        new_row[f'building type_{config}'] = features.get(f'building type_{config}')
        new_row[f'area low {config}'] = features.get(f'area low {config}')
        new_row[f'area high {config}'] = features.get(f'area high {config}')
        new_row[f'price low {config}'] = features.get(f'price low {config}')
        new_row[f'price high {config}'] = features.get(f'price high {config}')

    data_refined.append(new_row)

df_final_refined_v2 = pd.DataFrame(data_refined).set_index('PropertyName')

In [ ]:
df_final_refined_v2['building type_Land'] = df_final_refined_v2['building type_Land'].replace({'':'Land'})

In [ ]:
df['PriceDetails'][10]

"{'2 BHK': {'building_type': 'Independent Floor', 'area_type': 'Carpet Area', 'area': '1,055 sq.ft.', 'price-range': '₹ 1.05 - 1.5 Cr'}, '3 BHK': {'building_type': 'Independent Floor', 'area_type': 'Carpet Area', 'area': '1,325 - 1,525 sq.ft.', 'price-range': '₹ 1.35 - 1.84 Cr'}}"

In [ ]:
df_final_refined_v2

,building type_1 BHK,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,building type_2 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,building type_3 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,building type_4 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,building type_5 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,building type_6 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,building type_1 RK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,building type_Land,area low Land,area high Land,price low Land,price high Land
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,None,NaN,NaN,NaN,NaN,Apartment,1370.00,1370.00,2.0000,2.4000,Apartment,1850.00,2050.00,2.2500,3.59,Apartment,2600.00,2600.00,3.24,4.56,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
M3M Crown,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Apartment,1605.00,2170.00,2.2000,3.03,Apartment,2248.00,2670.00,3.08,3.73,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Adani Brahma Samsara Vilasa,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Independent Floor,1800.00,3150.00,2.4300,15.75,Independent Floor,2750.00,4500.00,3.36,22.50,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Land,500.00,4329.00,2.0500,41.1300
Sobha City,None,NaN,NaN,NaN,NaN,Apartment,1381.00,1692.00,1.5500,3.2100,Apartment,1711.00,2343.00,1.7600,4.79,Apartment,2423.00,2963.00,2.50,6.06,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Signature Global City 93,None,NaN,NaN,NaN,NaN,Independent Floor,981.00,1118.00,0.9301,1.0600,Independent Floor,1235.00,1530.00,1.1200,1.45,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Whiteland The Aspen,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Apartment,2290.00,2937.00,3.0700,3.96,Apartment,3243.00,12613.00,4.35,17.39,Apartment,6096.00,7582.00,8.21,10.24,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Bestech Altura,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Apartment,2015.00,2150.00,1.5300,1.85,Apartment,2250.00,2675.00,1.71,2.46,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Elan The Presidential,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,Apartment,1347.00,2600.00,4.0500,8.88,Apartment,1692.00,3100.00,4.89,10.59,Apartment,2275.00,4100.00,6.00,14.00,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN
Signature Global City 92,None,NaN,NaN,NaN,NaN,Independent Floor,959.00,1145.00,1.2600,1.5100,Independent Floor,1091.00,1535.00,0.8646,4.01,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN


In [ ]:
categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()

In [ ]:
categorical_columns

['building type_1 BHK',
 'building type_2 BHK',
 'building type_3 BHK',
 'building type_4 BHK',
 'building type_5 BHK',
 'building type_6 BHK',
 'building type_1 RK',
 'building type_Land']

In [ ]:
ohe_df = pd.get_dummies(df_final_refined_v2, columns=categorical_columns, drop_first=True)

In [ ]:
ohe_df.fillna(0,inplace=True)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
ohe_df_normalized = pd.DataFrame(scaler.fit_transform(ohe_df), columns=ohe_df.columns, index=ohe_df.index)

In [ ]:
ohe_df_normalized.head()

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,price low 3 BHK,price high 3 BHK,area low 4 BHK,area high 4 BHK,price low 4 BHK,price high 4 BHK,area low 5 BHK,area high 5 BHK,price low 5 BHK,price high 5 BHK,area low 6 BHK,area high 6 BHK,price low 6 BHK,price high 6 BHK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,area low Land,area high Land,price low Land,price high Land,building type_1 BHK_Service Apartment,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,-0.252814,-0.16994,-0.105414,-0.082501,1.228676,1.023897,-0.174239,1.159482,0.554181,0.370414,0.807279,0.514197,0.602763,0.211872,0.384340,0.241949,-0.470121,-0.461601,-0.248586,-0.236423,-0.125842,-0.11918,-0.077809,-0.073538,-0.105374,-0.102742,-0.090707,-0.082895,-0.448139,-0.372283,-0.196117,-0.241008,-0.11134,-0.289950,-0.064018,-0.373544,-0.064018,-0.171499,-0.255377,-0.236716,-0.11134,-0.216815,-0.064018
M3M Crown,-0.252814,-0.16994,-0.105414,-0.082501,-0.889440,-0.892604,-0.283860,-0.384982,0.293948,0.472092,0.772337,0.342389,0.383088,0.242907,0.342473,0.108874,-0.470121,-0.461601,-0.248586,-0.236423,-0.125842,-0.11918,-0.077809,-0.073538,-0.105374,-0.102742,-0.090707,-0.082895,-0.448139,-0.372283,-0.196117,-0.241008,-0.11134,-0.289950,-0.064018,-0.373544,-0.064018,-0.171499,-0.255377,-0.236716,-0.11134,-0.216815,-0.064018
Adani Brahma Samsara Vilasa,-0.252814,-0.16994,-0.105414,-0.082501,-0.889440,-0.892604,-0.283860,-0.384982,0.501072,1.302463,0.933074,4.244886,0.696375,1.054264,0.415740,3.118279,-0.470121,-0.461601,-0.248586,-0.236423,-0.125842,-0.11918,-0.077809,-0.073538,-0.105374,-0.102742,-0.090707,-0.082895,0.367111,1.823782,0.641818,7.620270,-0.11134,-0.289950,-0.064018,2.677063,-0.064018,-0.171499,3.915780,-0.236716,-0.11134,-0.216815,-0.064018
Sobha City,-0.252814,-0.16994,-0.105414,-0.082501,1.245683,1.474344,-0.198904,1.680738,0.406539,0.618678,0.464840,0.882357,0.492302,0.372813,0.190707,0.482444,-0.470121,-0.461601,-0.248586,-0.236423,-0.125842,-0.11918,-0.077809,-0.073538,-0.105374,-0.102742,-0.090707,-0.082895,-0.448139,-0.372283,-0.196117,-0.241008,-0.11134,-0.289950,-0.064018,-0.373544,-0.064018,-0.171499,-0.255377,-0.236716,-0.11134,-0.216815,-0.064018
Signature Global City 93,-0.252814,-0.16994,-0.105414,-0.082501,0.627254,0.671373,-0.232881,0.297156,-0.099056,-0.070191,0.017571,-0.142355,-1.019837,-0.940875,-0.463459,-0.489159,-0.470121,-0.461601,-0.248586,-0.236423,-0.125842,-0.11918,-0.077809,-0.073538,-0.105374,-0.102742,-0.090707,-0.082895,-0.448139,-0.372283,-0.196117,-0.241008,-0.11134,3.448875,-0.064018,2.677063,-0.064018,-0.171499,-0.255377,-0.236716,-0.11134,-0.216815,-0.064018


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix
cosine_sim2 = cosine_similarity(ohe_df_normalized)

In [ ]:
cosine_sim2.shape

(245, 245)

In [ ]:
def recommend_properties_with_scores(property_name, top_n=247):

    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim2[ohe_df_normalized.index.get_loc(property_name)]))

    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]

    # Retrieve the names of the top properties using the indices
    top_properties = ohe_df_normalized.index[top_indices].tolist()

    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })

    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('M3M Golf Hills')

,PropertyName,SimilarityScore
0,AIPL The Peaceful Homes,0.955808
1,Smartworld One DXP,0.954996
2,Unitech Escape,0.953471
3,BPTP Terra,0.943627
4,Sobha City,0.929143
5,Unitech Harmony,0.925606
6,Corona Optus,0.919820
7,Puri Emerald Bay,0.917933
8,Ireo Skyon,0.916509
9,Vatika Seven Elements,0.912943


In [ ]:
df[['PropertyName','LocationAdvantages']]['LocationAdvantages'][0]

"{'bajghera road': '800 Meter', 'palam vihar halt': '2.5 KM', 'dpsg palam vihar': '3.1 KM', 'park hospital': '3.1 KM', 'gurgaon old railway station': '4.9 KM', 'the northcap university': '5.4 KM', 'dwarka expy': '1.2 KM', 'hyatt place gurgaon udyog vihar': '7.7 KM', 'dwarka sector 21 metro station': '7.2 KM', 'pacific d21 mall': '7.4 KM', 'indira gandhi international airport': '14.7 KM', 'hamoni golf camp': '6.2 KM', 'fun n food waterpark': '8.8 KM', 'accenture ddc5': '9 KM'}"

In [ ]:
def distance_to_meters(distance_str):
    try:
        if 'Km' in distance_str or 'KM' in distance_str:
            return float(distance_str.split()[0]) * 1000
        elif 'Meter' in distance_str or 'meter' in distance_str:
            return float(distance_str.split()[0])
        else:
            return None
    except:
        return None

In [ ]:
# Extract distances for each location
location_matrix = {}
for index, row in df.iterrows():
    distances = {}
    for location, distance in ast.literal_eval(row['LocationAdvantages']).items():
        distances[location] = distance_to_meters(distance)
    location_matrix[index] = distances

# Convert the dictionary to a dataframe
location_df = pd.DataFrame.from_dict(location_matrix, orient='index')

# Display the first few rows
location_df.head()

,bajghera road,palam vihar halt,dpsg palam vihar,park hospital,gurgaon old railway station,the northcap university,dwarka expy,hyatt place gurgaon udyog vihar,dwarka sector 21 metro station,pacific d21 mall,indira gandhi international airport,hamoni golf camp,fun n food waterpark,accenture ddc5,palam vihar halt railway station,dwaraka expressway,tau devilal sports complex,hyatt place,altrade business centre,aipl business club,heritage xperiential learning school,ck birla hospital,paras trinity mall,sector 55 56 rapid metro station,de adventure park,golf course extension rd,doubletree by hilton hotel,kiit college of engineering sohna road,mehrauli gurgaon rd,nirvana rd,teri golf course,the shikshiyan school,wtc plaza,luxus haritma resort,bsf golf course,rion's hospital,gurgaon,dwarka sector 21,nehru stadium,vasant kunj,pranavananda international school,dlf site central office,holiday inn sector 90,krishna hospital,royal institute of science & management,sapphire 93 mall,nh48,garhi harsaru junction,manesar golf course,aapno ghar,vega school,dlf corporate greens,miracles apollo cradle /spectra hospital,hyatt regency,nh 248,golden greens golf & resorts limited,mount olympus junior school,miracles apollo hospital,nh 8,savoy suites,imt manesar,amity university,dwarka expy sector 109,euro international school sector 51,jai sai ram hospital,aryan hospital,idea cosmic plaza,pataudi road,s n international school,aarvy healthcare super speciality hospital,iris broadway mall,imperia mindspace,aipl business tower,heritage school,lotus valley international school,gurugram university,omaxe city centre mall,sushant university,badshahpur sohna rd hwysector 48,naurangpur cricket stadium,naurangpur road,national highway 8,vatika town square inxt,ompee international school,manesar bus stand,yashlok medical centre,euro international school,worldmark,capital cyberscape,the shriram millennium school,badshahpur sohna rd hwy,nakhrola stadium,delhi jaipur expressway,bal bharati public school,vatika business centre,st xavier high school,miracles apollo cradle,ambience mall,nh8,delhi public school,elan miracle mall,agri business management collage,grand hyatt gurgaon,duke horse riding club,pvr drive in cinema,w pratiksha hospital,metro world mall,unicosmos school,faridabad gurgaon rd,sohna road,bestech business tower,appu ghar water park,skyjumper trampoline park,axis bank,kmp expressway,karma lakelands,jungle safari & trails,dps manesar,medanta hospital,lingaya's lalita devi institute,asf insignia sez,banjara market gurugram,central plaza mall,paras hospital,indian school of hospitality,vatika city centre,aatish hospital,info technology park phase 2,huda metro station,southern peripheral rd,global ways school,radisson hotel,nh 248 a,mavens inn,sanar international hospital,sector 53 54 metro station,iilm university,the banyan tree world school,the big tree cafe,dlf golf and country club,arvy hospital,dpg degree college,gurgaon dreamz mall,metro hospital palam vihar,delhi ajmer expressway,infinity business park,bijwasan railway station,heritage xperiential learning crpf rd,sector 54 chowk metro station,gurgaon delhi expy,genesis hospital,dpgitm engineering college,sapphire 83 mall,holiday inn hotel sector 90,national tennis academy sector 98,nh 8 delhi jaipur highway,nouveau medics multispeciality opd,heritage village resort & spa,sector 86 road,delh ajmer expy,medanta the medicity,airia mall,amma hospital,dps international edge,eros city square,splendor trade tower,narayana junior college,dharampur main road,oyster's water park,vivanta dwarka new delhi,dlf corporate park,g d goenka world school,kr mangalam university,smart view hotel and resort,central park flower valley,shopping complex,central peripheral road,nakhrola stadium sector 81a,nh 08,gd goenka university,fly india adventure resort,vardaan hospital,imt office sohna,jms marine square mall,vibrant hospital,prime scholars international school,ramgarh farms & resort,basai road,shree krishna hospita

In [ ]:
location_df.index = df.PropertyName

In [ ]:
location_df.head()

,bajghera road,palam vihar halt,dpsg palam vihar,park hospital,gurgaon old railway station,the northcap university,dwarka expy,hyatt place gurgaon udyog vihar,dwarka sector 21 metro station,pacific d21 mall,indira gandhi international airport,hamoni golf camp,fun n food waterpark,accenture ddc5,palam vihar halt railway station,dwaraka expressway,tau devilal sports complex,hyatt place,altrade business centre,aipl business club,heritage xperiential learning school,ck birla hospital,paras trinity mall,sector 55 56 rapid metro station,de adventure park,golf course extension rd,doubletree by hilton hotel,kiit college of engineering sohna road,mehrauli gurgaon rd,nirvana rd,teri golf course,the shikshiyan school,wtc plaza,luxus haritma resort,bsf golf course,rion's hospital,gurgaon,dwarka sector 21,nehru stadium,vasant kunj,pranavananda international school,dlf site central office,holiday inn sector 90,krishna hospital,royal institute of science & management,sapphire 93 mall,nh48,garhi harsaru junction,manesar golf course,aapno ghar,vega school,dlf corporate greens,miracles apollo cradle /spectra hospital,hyatt regency,nh 248,golden greens golf & resorts limited,mount olympus junior school,miracles apollo hospital,nh 8,savoy suites,imt manesar,amity university,dwarka expy sector 109,euro international school sector 51,jai sai ram hospital,aryan hospital,idea cosmic plaza,pataudi road,s n international school,aarvy healthcare super speciality hospital,iris broadway mall,imperia mindspace,aipl business tower,heritage school,lotus valley international school,gurugram university,omaxe city centre mall,sushant university,badshahpur sohna rd hwysector 48,naurangpur cricket stadium,naurangpur road,national highway 8,vatika town square inxt,ompee international school,manesar bus stand,yashlok medical centre,euro international school,worldmark,capital cyberscape,the shriram millennium school,badshahpur sohna rd hwy,nakhrola stadium,delhi jaipur expressway,bal bharati public school,vatika business centre,st xavier high school,miracles apollo cradle,ambience mall,nh8,delhi public school,elan miracle mall,agri business management collage,grand hyatt gurgaon,duke horse riding club,pvr drive in cinema,w pratiksha hospital,metro world mall,unicosmos school,faridabad gurgaon rd,sohna road,bestech business tower,appu ghar water park,skyjumper trampoline park,axis bank,kmp expressway,karma lakelands,jungle safari & trails,dps manesar,medanta hospital,lingaya's lalita devi institute,asf insignia sez,banjara market gurugram,central plaza mall,paras hospital,indian school of hospitality,vatika city centre,aatish hospital,info technology park phase 2,huda metro station,southern peripheral rd,global ways school,radisson hotel,nh 248 a,mavens inn,sanar international hospital,sector 53 54 metro station,iilm university,the banyan tree world school,the big tree cafe,dlf golf and country club,arvy hospital,dpg degree college,gurgaon dreamz mall,metro hospital palam vihar,delhi ajmer expressway,infinity business park,bijwasan railway station,heritage xperiential learning crpf rd,sector 54 chowk metro station,gurgaon delhi expy,genesis hospital,dpgitm engineering college,sapphire 83 mall,holiday inn hotel sector 90,national tennis academy sector 98,nh 8 delhi jaipur highway,nouveau medics multispeciality opd,heritage village resort & spa,sector 86 road,delh ajmer expy,medanta the medicity,airia mall,amma hospital,dps international edge,eros city square,splendor trade tower,narayana junior college,dharampur main road,oyster's water park,vivanta dwarka new delhi,dlf corporate park,g d goenka world school,kr mangalam university,smart view hotel and resort,central park flower valley,shopping complex,central peripheral road,nakhrola stadium sector 81a,nh 08,gd goenka university,fly india adventure resort,vardaan hospital,imt office sohna,jms marine square mall,vibrant hospital,prime scholars international school,ramgarh farms & resort,basai road,shree krishna hospita

In [ ]:

location_df.fillna(54000,inplace=True)

In [ ]:
location_df.head()

,bajghera road,palam vihar halt,dpsg palam vihar,park hospital,gurgaon old railway station,the northcap university,dwarka expy,hyatt place gurgaon udyog vihar,dwarka sector 21 metro station,pacific d21 mall,indira gandhi international airport,hamoni golf camp,fun n food waterpark,accenture ddc5,palam vihar halt railway station,dwaraka expressway,tau devilal sports complex,hyatt place,altrade business centre,aipl business club,heritage xperiential learning school,ck birla hospital,paras trinity mall,sector 55 56 rapid metro station,de adventure park,golf course extension rd,doubletree by hilton hotel,kiit college of engineering sohna road,mehrauli gurgaon rd,nirvana rd,teri golf course,the shikshiyan school,wtc plaza,luxus haritma resort,bsf golf course,rion's hospital,gurgaon,dwarka sector 21,nehru stadium,vasant kunj,pranavananda international school,dlf site central office,holiday inn sector 90,krishna hospital,royal institute of science & management,sapphire 93 mall,nh48,garhi harsaru junction,manesar golf course,aapno ghar,vega school,dlf corporate greens,miracles apollo cradle /spectra hospital,hyatt regency,nh 248,golden greens golf & resorts limited,mount olympus junior school,miracles apollo hospital,nh 8,savoy suites,imt manesar,amity university,dwarka expy sector 109,euro international school sector 51,jai sai ram hospital,aryan hospital,idea cosmic plaza,pataudi road,s n international school,aarvy healthcare super speciality hospital,iris broadway mall,imperia mindspace,aipl business tower,heritage school,lotus valley international school,gurugram university,omaxe city centre mall,sushant university,badshahpur sohna rd hwysector 48,naurangpur cricket stadium,naurangpur road,national highway 8,vatika town square inxt,ompee international school,manesar bus stand,yashlok medical centre,euro international school,worldmark,capital cyberscape,the shriram millennium school,badshahpur sohna rd hwy,nakhrola stadium,delhi jaipur expressway,bal bharati public school,vatika business centre,st xavier high school,miracles apollo cradle,ambience mall,nh8,delhi public school,elan miracle mall,agri business management collage,grand hyatt gurgaon,duke horse riding club,pvr drive in cinema,w pratiksha hospital,metro world mall,unicosmos school,faridabad gurgaon rd,sohna road,bestech business tower,appu ghar water park,skyjumper trampoline park,axis bank,kmp expressway,karma lakelands,jungle safari & trails,dps manesar,medanta hospital,lingaya's lalita devi institute,asf insignia sez,banjara market gurugram,central plaza mall,paras hospital,indian school of hospitality,vatika city centre,aatish hospital,info technology park phase 2,huda metro station,southern peripheral rd,global ways school,radisson hotel,nh 248 a,mavens inn,sanar international hospital,sector 53 54 metro station,iilm university,the banyan tree world school,the big tree cafe,dlf golf and country club,arvy hospital,dpg degree college,gurgaon dreamz mall,metro hospital palam vihar,delhi ajmer expressway,infinity business park,bijwasan railway station,heritage xperiential learning crpf rd,sector 54 chowk metro station,gurgaon delhi expy,genesis hospital,dpgitm engineering college,sapphire 83 mall,holiday inn hotel sector 90,national tennis academy sector 98,nh 8 delhi jaipur highway,nouveau medics multispeciality opd,heritage village resort & spa,sector 86 road,delh ajmer expy,medanta the medicity,airia mall,amma hospital,dps international edge,eros city square,splendor trade tower,narayana junior college,dharampur main road,oyster's water park,vivanta dwarka new delhi,dlf corporate park,g d goenka world school,kr mangalam university,smart view hotel and resort,central park flower valley,shopping complex,central peripheral road,nakhrola stadium sector 81a,nh 08,gd goenka university,fly india adventure resort,vardaan hospital,imt office sohna,jms marine square mall,vibrant hospital,prime scholars international school,ramgarh farms & resort,basai road,shree krishna hospita

In [ ]:
import pickle
pickle.dump(location_df,open('location_distance.pkl','wb'))

In [ ]:
from sklearn.preprocessing import StandardScaler
# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
location_df_normalized = pd.DataFrame(scaler.fit_transform(location_df), columns=location_df.columns, index=location_df.index)

In [ ]:
cosine_sim3 = cosine_similarity(location_df_normalized)

In [ ]:
def recommend_properties_with_scores(property_name, top_n=247):

    cosine_sim_matrix = 30*cosine_sim1 + 20*cosine_sim2 + 8*cosine_sim3
    # cosine_sim_matrix = cosine_sim3

    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))

    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]

    # Retrieve the names of the top properties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()

    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })

    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('Ireo Victory Valley')

,PropertyName,SimilarityScore
0,Pioneer Urban Presidia,27.793198
1,Ambience Creacions,27.753581
2,DLF The Crest,24.160151
3,Pioneer Araya,23.320170
4,AIPL The Peaceful Homes,21.451313
5,Silverglades The Melia,21.090832
6,SS The Leaf,20.799605
7,Adani M2K Oyster Grande,20.314877
8,Tulip Monsella,19.823547
9,Indiabulls Centrum Park,19.657220


In [ ]:
pickle.dump(cosine_sim1,open('cosine_sim1.pkl','wb'))
pickle.dump(cosine_sim2,open('cosine_sim2.pkl','wb'))
pickle.dump(cosine_sim3,open('cosine_sim3.pkl','wb'))

In [ ]:
import pandas as pd
import re
import ast

rows = []

for _, row in df.iterrows():

    society = row["PropertyName"]

    # string -> dictionary
    price_details = ast.literal_eval(row["PriceDetails"])

    for bhk, details in price_details.items():

        # ---------- BHK ----------
        if bhk.lower() == "land":
            bhk_num = 0
        else:
            m = re.search(r"\d+", bhk)
            if m is None:
                continue
            bhk_num = int(m.group())

        # ---------- Price ----------
        price_range = details["price-range"]

        nums = re.findall(r"[\d.]+", price_range)

        if len(nums) == 0:
            min_price = max_price = None

        elif len(nums) == 1:
            min_price = max_price = float(nums[0])

        else:
            min_price = float(nums[0])
            max_price = float(nums[1])

        # ---------- Area ----------
        area = details["area"]

        area_nums = re.findall(r"[\d,]+", area)

        if len(area_nums) == 0:
            area_min = area_max = None

        elif len(area_nums) == 1:
            area_min = area_max = float(area_nums[0].replace(",", ""))

        else:
            area_min = float(area_nums[0].replace(",", ""))
            area_max = float(area_nums[1].replace(",", ""))

        rows.append({
            "society": society,
            "bhk": bhk_num,
            "price_min": min_price,
            "price_max": max_price,
            "area_min": area_min,
            "area_max": area_max
        })

price_bhk_df = pd.DataFrame(rows)

print(price_bhk_df.shape)
price_bhk_df.head()

(571, 6)


,society,bhk,price_min,price_max,area_min,area_max
0,Smartworld One DXP,2,2.00,2.40,1370.0,1370.0
1,Smartworld One DXP,3,2.25,3.59,1850.0,2050.0
2,Smartworld One DXP,4,3.24,4.56,2600.0,2600.0
3,M3M Crown,3,2.20,3.03,1605.0,2170.0
4,M3M Crown,4,3.08,3.73,2248.0,2670.0


In [ ]:
price_bhk_df = price_bhk_df[[
    "society",
    "bhk",
    "price_min",
    "price_max"
]]

In [ ]:
import pickle

pickle.dump(price_bhk_df, open("price_bhk_df.pkl", "wb"))

In [ ]:
cosine_sim1.shape
cosine_sim2.shape
cosine_sim3.shape

location_df.head()

df.head()

df.columns

(245, 245)

In [ ]:
price_bhk_df.head()



,society,bhk,price_min,price_max
0,Smartworld One DXP,2,2.00,2.40
1,Smartworld One DXP,3,2.25,3.59
2,Smartworld One DXP,4,3.24,4.56
3,M3M Crown,3,2.20,3.03
4,M3M Crown,4,3.08,3.73


In [ ]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,FacilitiesStr
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'bajghera road': '800 Meter', 'palam vihar ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Salon, Restaurant, Spa, Cafete...",Swimming Pool Salon Restaurant Spa Cafeteria S...
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'dpsg palam vihar': '1.4 Km', 'the northcap u...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","[Bowling Alley, Mini Theatre, Manicured Garden...",Bowling Alley Mini Theatre Manicured Garden Sw...
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'aipl business club': '2.7 Km', 'heritage xpe...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,"[Terrace Garden, Gazebo, Fountain, Amphitheatr...",Terrace Garden Gazebo Fountain Amphitheatre Pa...
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'the shikshiyan school': '2.9 KM', 'wtc plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Swimming Pool, Volley Ball Court, Aerobics Ce...",Swimming Pool Volley Ball Court Aerobics Centr...
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'pranavananda international school': '450 m',...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,"[Mini Theatre, Doctor on Call, Concierge Servi...",Mini Theatre Doctor on Call Concierge Service ...


In [ ]:
location_distance.head()

In [ ]:
pickle.dump(df, open("df_recom.pkl", "wb"))